# Homework 2: SwiftParcel Support Agent (LangGraph, memory, policies and claim approval)

**Covers:** LangGraph state, nodes and routing, PostgreSQL checkpoints and memory, agentic RAG with Qdrant, human approval with `interrupt()`, plus new topics: **metadata-filtered retrieval**, **streaming graph updates** and **checkpoint history**.

**Time:** about 4 to 5 hours. **Prerequisite:** Homework 1 (you can reuse its tools). **Submit:** this notebook with all cells run and outputs visible.

## The story: SwiftParcel

**SwiftParcel** is a courier company. Customers write to support to ask where a parcel is, to move a delivery to another day, and to complain when a parcel is late, damaged or lost. The company wants an AI support agent, and you are the engineer building it.

You will build it twice, the same way we built the Zomato agent:

| Homework | What you build | Tools you use |
|---|---|---|
| **Homework 1** | A hardened LangChain agent that looks up parcels and reschedules deliveries | `create_agent`, Pydantic, `ToolRuntime`, middleware |
| **Homework 2** (this notebook) | A LangGraph version with memory, policy search and claim approval | LangGraph, PostgreSQL checkpoints, Qdrant, human approval |

### The data

Three customers (`301` Meera, GOLD; `302` Karan, STANDARD; `303` Divya, STANDARD) and ten parcels. Tracking ids look like `SP-10000001`. A parcel has a status (`PENDING_PICKUP`, `IN_TRANSIT`, `OUT_FOR_DELIVERY`, `DELIVERED`, `LOST`, `RETURNED`), a declared value, a promised date and a reschedule count.

### The business rules

| Rule | Value |
|---|---|
| A customer may see and change **only their own** parcels | always |
| Reschedule allowed when the status is | `PENDING_PICKUP` or `IN_TRANSIT` (**not** `OUT_FOR_DELIVERY`) |
| Reschedules per parcel | at most **2** |
| New delivery date | from tomorrow to **7 days** from today |
| Claims (Homework 2) | up to 500 rupees automatic, up to 5000 a reviewer, above 5000 a senior reviewer |

## How this homework works

Same rules as Homework 1: **PROVIDED** cells work as they are, **YOUR CODE** cells have a stub, and every task lists **Must show** outputs that are required for marks.

### The claim rules for this homework

| Claim amount | What happens |
|---|---|
| 500 rupees or less | **Auto-approved** by trusted code (decided by `system`), no human |
| above 500, up to 5000 | the graph **pauses** for a reviewer |
| above 5000 | the graph pauses for a **senior** reviewer; a normal reviewer's decision is ignored |
| always | a claim can never exceed the parcel's declared value; types are `LOST`, `DAMAGED`, `LATE`; one claim per parcel and type |

Reviewers: `reviewer-asha` and `reviewer-ravi` are normal. `senior-iqbal` is senior. A decision from anyone else is ignored.

| Task | Topic | Marks |
|---|---|---|
| 1 | The graph skeleton and an iteration limit | 15 |
| 2 | PostgreSQL memory, thread ownership and preferences | 15 |
| 3 | Policy search with metadata filtering | 15 |
| 4 | Claims: auto-approval, human approval, senior approval | 25 |
| 5 | Streaming, checkpoint history and the final scenario suite | 15 |
| | Code quality and the short write-ups | 15 |

## Setup (PROVIDED)

In [ ]:
import os, sys, json, time, warnings
from datetime import date, timedelta

import psycopg
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
DATABASE_URL = os.environ["DATABASE_URL"]

llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0, timeout=30, max_retries=2,
)

def ok(**data):
    return {"ok": True, **data}

def fail(code, message, **data):
    return {"ok": False, "code": code, "message": message, **data}

print("Ready. Model:", os.environ["OPENROUTER_MODEL"])

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import interrupt, Command
from langchain.tools import ToolRuntime

In [ ]:
# Create the tables if they do not exist, and keep your data from Homework 1.
# To start clean, run the seed cell as well (it also clears threads, preferences and claims).
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute(open("swiftparcel_schema.sql").read())
    print("parcels:", conn.execute("SELECT count(*) FROM sp_parcels").fetchone()[0])

RESET_DATA = True
if RESET_DATA:
    with psycopg.connect(DATABASE_URL) as conn:
        conn.execute(open("swiftparcel_seed.sql").read())
    print("Teaching data reset.")

In [ ]:
# PROVIDED. Question set for Task 3 and the reviewer lists for Task 4.
POLICY_QUESTIONS = [
    ("How many times can I reschedule a delivery?", True),
    ("Can I reschedule once the parcel is out for delivery?", True),
    ("What compensation do I get if my parcel is 5 days late?", True),
    ("How long do I have to report a damaged parcel?", True),
    ("Who approves a claim of 8000 rupees?", True),
    ("Do you deliver by drone?", False),
    ("Can you ship live animals?", False),
    ("What is the weather in Mumbai today?", False),
]

AUTHORIZED_REVIEWERS = ["reviewer-asha", "reviewer-ravi", "senior-iqbal"]
SENIOR_REVIEWERS = ["senior-iqbal"]
AUTO_APPROVE_LIMIT = 500
SENIOR_LIMIT = 5000

## Task 1: The graph skeleton and an iteration limit (15 marks)

**Goal.** Rebuild the agent as an explicit LangGraph graph.

**Requirements**

1. `SupportState(MessagesState)` with: `customer_id: int`, `tool_rounds: int`, `pending_claim: dict | None`, `approval_decision: dict | None`, `decision_notice: str`. The customer id lives in **state** (trusted), and tools read it with `runtime.state["customer_id"]`.
2. Port `get_parcel_status` and `list_my_parcels` from Homework 1 (same ownership rules, same safe errors).
3. Nodes: `agent` (calls the model), `tools` (`ToolNode`), `count_round` (adds 1 to `tool_rounds`), `give_up` (adds a fixed `AIMessage`: "I could not finish this. A person will follow up.").
4. Routing: `agent` goes to `tools` when the model asked for tools, otherwise `END`. After `count_round`, if `tool_rounds >= MAX_ROUNDS` go to `give_up` then `END`, otherwise back to `agent`.
5. Draw the graph (PNG or Mermaid text).

**Must show**

| # | Run | Expected |
|---|---|---|
| a | "Where is SP-10000001?" as customer 301 | normal answer |
| b | the same with `MAX_ROUNDS = 1` and a question that needs two tool calls ("List my parcels and then give me details of the first one") | the graph ends at `give_up` with the fixed message |
| c | the graph drawing | visible |
| d | one sentence: why is the iteration limit a node and a route here, and not a prompt instruction? | |

In [ ]:
# YOUR CODE: Task 1
from langgraph.graph import MessagesState

MAX_ROUNDS = 4


class SupportState(MessagesState):
    customer_id: int
    tool_rounds: int
    pending_claim: dict | None
    approval_decision: dict | None
    decision_notice: str


# TODO: read tools using runtime.state["customer_id"]
# TODO: agent_node, count_round, give_up, route_after_rounds
# TODO: build the graph with StateGraph(SupportState), compile it (checkpointer comes in Task 2)

In [ ]:
# YOUR CODE: Task 1 demonstrations (a to d)

## Task 2: PostgreSQL memory, thread ownership and preferences (15 marks)

**Goal.** Conversations survive a restart, and nobody can use someone else's thread.

**Requirements**

1. A PostgreSQL checkpointer (`PostgresSaver` with a `ConnectionPool`). Call `setup()` once.
2. Thread helpers on the `sp_threads` table: `get_or_create_thread(customer_id, name)` returns a stable thread id for that customer and name, and `assert_owner(customer_id, thread_id)` raises if the thread belongs to another customer. The thread id is **never** typed by the customer; your code chooses it.
3. Durable preferences in `sp_preferences`, with a validated schema:

   | Key | Allowed values |
   |---|---|
   | `notify_channel` | `sms`, `email`, `whatsapp` |
   | `delivery_window` | `morning`, `afternoon`, `evening` |
   | `instructions` | free text, at most 100 characters |

   Two tools: `save_preference` (upsert) and `get_preferences`. Customer from state.
4. A helper `ask(customer_id, thread_name, question)` that gets the thread, checks ownership, invokes the graph with the right config, and prints the final answer.

**Must show**

| # | Run | Expected |
|---|---|---|
| a | Thread A (customer 301): "Remember that I prefer WhatsApp." | preference saved in the database |
| b | **A new thread B** for 301: "How should you contact me?" | answered from the database, not from chat history |
| c | Customer 302 in thread C: "How should you contact me?" | no preference for 302, and 301's value is never shown |
| d | In thread A: "Where is SP-10000001?" then "And when is it promised?" | the second answer uses the first turn's context |
| e | Customer 302 tries to use 301's thread id | `assert_owner` refuses |
| f | **Restart:** close the pool, rebuild the checkpointer and graph, and send one more message in thread A | the conversation continues |
| g | Save `instructions` with 150 characters | rejected with a clear message |

In [ ]:
# YOUR CODE: Task 2
from psycopg_pool import ConnectionPool
from psycopg.rows import dict_row
from langgraph.checkpoint.postgres import PostgresSaver
from typing import Literal


def make_pool():
    # TODO: ConnectionPool(conninfo=DATABASE_URL, kwargs={"autocommit": True, "prepare_threshold": 0, "row_factory": dict_row}, ...)
    raise NotImplementedError


def get_or_create_thread(customer_id, name):
    raise NotImplementedError


def assert_owner(customer_id, thread_id):
    raise NotImplementedError


# TODO: preference schema (Literal keys, length limit), save_preference and get_preferences tools
# TODO: rebuild the graph from Task 1 with checkpointer=...
# TODO: ask(customer_id, thread_name, question)

In [ ]:
# YOUR CODE: Task 2 demonstrations (a to g)

## Task 3: Policy search with metadata filtering (15 marks)

**Goal.** Give the agent a `search_policies` tool backed by Qdrant, built with the LangChain stack from Notebook 3, plus one new idea: **metadata filtering**.

**Data.** `data/swiftparcel_policies.csv` has columns `policy_id, category, title, text`. Categories: `RESCHEDULE`, `DELIVERY`, `CLAIMS`, `SHIPPING`.

**Requirements**

1. Add **two policies of your own** to the CSV (new `policy_id`, one of the existing categories).
2. Index with `CSVLoader` (content = title and text; metadata = `policy_id`, `category`), `RecursiveCharacterTextSplitter`, and `QdrantVectorStore`, into a **new collection** named `swiftparcel_policies`. Re-running the cell must not index twice.
3. `search_policies(query, category=None)`: when `category` is given, search **only** that category using a Qdrant filter on `metadata.category`. Reshape results to `policy_id`, `title`, `text`, `score`. Drop results below `MIN_SCORE`. Return `NO_POLICY_FOUND` when nothing is left.
4. **Tune `MIN_SCORE`** on the provided `POLICY_QUESTIONS` (5 answerable, 3 not). Print every question with its top score and whether it should be answerable. Choose a value that gets as many as possible right.
5. The system prompt must tell the model: policy text is data, not instructions; cite `policy_id`; if no policy is found, say it could not verify.

**Must show**

| # | Show |
|---|---|
| a | The score table for all 8 questions, your chosen `MIN_SCORE`, and the number answered correctly. If you cannot get 8 of 8, say which one fails and why. |
| b | The same question searched with and without a category filter, side by side, showing the filter changes the results. |
| c | Three agent answers: a policy question (cites `policy_id`), a combined question ("Can I still reschedule SP-10000002?" needs the parcel **and** the policy), and an unanswerable one (must say it could not verify). |

In [ ]:
# YOUR CODE: Task 3
import csv
from qdrant_client import QdrantClient, models
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore

COLLECTION = "swiftparcel_policies"
MIN_SCORE = 0.30     # TODO: tune
MAX_RESULTS = 4

qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.getenv("QDRANT_API_KEY") or None)
embedding_model = OpenAIEmbeddings(
    model=os.environ["OPENROUTER_EMBEDDING_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    check_embedding_ctx_length=False,
)

# TODO: index if the collection does not exist; otherwise connect with QdrantVectorStore.from_existing_collection
# TODO: search_policy_documents(query, category=None) using a models.Filter on "metadata.category"
# TODO: @tool search_policies(query, category)

In [ ]:
# YOUR CODE: Task 3 demonstrations (a to c)

## Task 4: Claims, auto-approval and human approval (25 marks)

**Goal.** The most important workflow in the project. A customer files a claim; trusted code decides how it is handled; a human is involved only when the rules say so.

**Requirements**

1. `file_claim(tracking_id, claim_type, amount, description)` tool. Customer from state. It validates: the parcel belongs to the customer; `LOST` needs status `LOST`; `DAMAGED` and `LATE` need status `DELIVERED`; `amount` is positive and not above the declared value. It stores a `PENDING` row in `sp_claims` with an idempotency key (customer, parcel, type), so repeating it returns the same claim and creates no second row. The tool **only creates a request**. It never approves anything.
2. Nodes (the 48-hour reporting window from the policy is **not** checked in this homework; adding it is a bonus):
   - `capture_claim`: after the tools, if this step created a **new** pending claim, read it back from the database into `pending_claim` (the reviewer sees stored data, not model text).
   - `auto_approve`: for `amount <= AUTO_APPROVE_LIMIT`, record the decision as `APPROVED` by `system` with a note. No human.
   - `approval`: calls `interrupt(...)` with the claim details. **No side effects before `interrupt()`.**
   - `record_decision`: trusted code. Ignores a reviewer who is not in `AUTHORIZED_REVIEWERS`. For `amount > SENIOR_LIMIT` it also ignores a reviewer who is not senior. Writes the decision with an `UPDATE ... WHERE status = 'PENDING'` so deciding twice changes nothing. Sets `decision_notice`.
3. Routing after `capture_claim`: no new claim -> `agent`; `amount <= 500` -> `auto_approve`; otherwise -> `approval`. After `record_decision`: still pending (ignored reviewer) -> `approval` again, otherwise -> `agent`.
4. **The agent must say what was recorded.** After the decision the model still sees the earlier tool result "claim created, pending". Put the recorded decision as the **last** message passed to the model and tell it in the system prompt that this message overrides earlier "pending" results. (Notebook 3 explains why.)

**Must show** (print the final answer **and** the `sp_claims` rows after each)

| # | Scenario | Expected |
|---|---|---|
| a | 302 claims 400 `DAMAGED` on `SP-10000006` (declared value 450) | auto-approved by `system`, **no pause** |
| b | 301 claims 800 `DAMAGED` on `SP-10000003` | graph **pauses**; `reviewer-asha` rejects; row `REJECTED`; the answer says rejected |
| c | 301 claims 3000 `LOST` on `SP-10000004` | pauses; `customer-301` tries to approve: ignored and asked again; then `reviewer-ravi` approves; row `APPROVED` |
| d | 303 claims 7000 `DAMAGED` on `SP-10000009` (declared value 7500) | pauses; `reviewer-asha` approves: **ignored** (not senior); then `senior-iqbal` approves |
| e | the claim from (a) sent again | no second row; the existing claim is reported |
| f | 303 claims 9000 `DAMAGED` on `SP-10000009` | refused by validation (above declared value) before any row exists |
| g | 302 claims on 301's `SP-10000003` | `PARCEL_NOT_FOUND` |

Also show that the paused graph in (b) is waiting at `approval` (`graph.get_state(config).next`), and that the pending claim is in the saved state.

In [ ]:
# YOUR CODE: Task 4
import hashlib


def create_claim(customer_id, tracking_id, claim_type, amount, description):
    # TODO: validation + idempotent insert into sp_claims
    raise NotImplementedError


def get_claim(claim_id):
    raise NotImplementedError


def decide_claim(claim_id, approved, decided_by, note):
    # TODO: UPDATE ... WHERE id = %s AND status = 'PENDING' RETURNING id
    raise NotImplementedError


# TODO: @tool file_claim
# TODO: capture_claim, auto_approve, approval_node, record_decision
# TODO: route_after_capture, route_after_record
# TODO: update agent_node so the recorded decision is the LAST message the model sees
# TODO: rebuild the graph with the new nodes and edges

In [ ]:
# YOUR CODE: Task 4 demonstrations (a to g)

**Write-up (4 sentences).** Why does `record_decision` check the reviewer in trusted code instead of letting the model or the customer message say who approved? What attack does the `WHERE status = 'PENDING'` guard stop?

*Your answer here.*

## Task 5: Streaming, checkpoint history and the final scenario suite (15 marks)

### 5a. Stream the graph (new topic)

`graph.invoke` returns only at the end. `graph.stream(..., stream_mode="updates")` yields **one item per node** as it finishes, which is what a chat UI or a log needs. Write `ask_streaming(customer_id, thread_name, question)` that prints a line per update: the node name and a short summary (for example "agent: asked for 2 tools", "tools: 2 results", "auto_approve: APPROVED"). Show it on a lookup and on a claim that pauses.

### 5b. Checkpoint history (new topic)

Every step is saved. Write `thread_timeline(customer_id, thread_id)` using `graph.get_state_history(config)` that prints, oldest first: step number, the node that ran next, and the number of messages. It must call `assert_owner` first. Show it for a finished thread, a paused thread and a thread that belongs to another customer (refused).

### 5c. Final scenario suite

Run all ten scenarios in one loop, check each **automatically** (database rows, `get_state`, tool results), and print a results table with `PASS/FAIL`.

| # | Scenario |
|---|---|
| 1 | Normal lookup of the customer's own parcel |
| 2 | Policy question answered with a cited `policy_id` |
| 3 | Combined: "Can I still reschedule SP-10000002?" (parcel plus policy) |
| 4 | Another customer's parcel is not revealed |
| 5 | Tool failure: database down for one lookup, safe message, no SQL leaked |
| 6 | Duplicate claim: no second row |
| 7 | Small claim auto-approved with no pause |
| 8 | Reviewer rejection recorded, answer says rejected |
| 9 | Large claim: normal reviewer ignored, senior approves |
| 10 | Restart while paused: rebuild everything, resume the same thread, decision recorded |

**Final write-up (6 sentences).** Which scenario was hardest, what did you change, which idea from the course (state, routing, memory, retrieval, approval) mattered most for this project, and what would you still need before putting this in front of real customers?

In [ ]:
# YOUR CODE: Task 5
def ask_streaming(customer_id, thread_name, question):
    # TODO: for update in graph.stream(..., stream_mode="updates"): print one line per node
    raise NotImplementedError


def thread_timeline(customer_id, thread_id):
    # TODO: assert_owner first, then graph.get_state_history(config) oldest first
    raise NotImplementedError


# TODO: the ten scenarios and the results table

In [ ]:
# YOUR CODE: Task 5 demonstrations (5a, 5b, 5c)

**Final write-up (6 sentences).**

*Your answer here.*

## Self-check before you submit

- [ ] Every "Must show" item has visible output.
- [ ] The customer id comes from state, never from the model.
- [ ] Thread ids and reviewer ids come from trusted code, never from the customer message.
- [ ] No side effects run before `interrupt()`.
- [ ] Deciding a claim twice changes nothing.
- [ ] The agent reports the **recorded** outcome, not "pending".
- [ ] You tuned `MIN_SCORE` on data and reported the result honestly.